# Master Notebook: Python Data Visualization & Plotting

**Purpose:** A standalone, practical reference for learning chart selection and plotting with Pandas, Matplotlib, Seaborn, and Plotly.

This notebook combines the core ideas from the supplied presentation and notebooks:
- Choosing a chart based on the analytical question
- Comparison, distribution, composition, relationship, trend, and geographical views
- The Python plotting ecosystem and when to use each library
- Formatting, saving, interpreting, and practicing with plots

**Data:** The examples create reproducible sample datasets in the notebook, so no external CSV/Excel file is required. You can replace them with your own DataFrames later.

## 1. Chart-selection guide

| Analytical question | Recommended chart | Example |
|---|---|---|
| Which category is largest? | Sorted bar / horizontal bar | Sales by product |
| How does a metric change over time? | Line chart | Monthly revenue |
| How are values distributed? | Histogram, KDE, box plot, violin plot | Salary distribution |
| How do two numeric variables move together? | Scatter plot, regression plot | Ad spend vs revenue |
| How is a total divided into parts? | Stacked bar; pie only for a few categories | Sales mix |
| How do groups differ across two categories? | Grouped bar, count plot | Sales by region and channel |
| How are numeric variables related overall? | Correlation heatmap / pair plot | Marketing metrics |
| How do values vary by geography? | Map if geographic coordinates or region boundaries exist; otherwise a regional bar chart | Sales by region |

**Rule of thumb:** Start with the business question and variable types, not with the chart that looks most attractive.

## 2. Plotting libraries: what each is good at

- **Pandas plotting:** quick exploratory charts directly from a Series or DataFrame.
- **Matplotlib:** detailed control over figure size, axes, labels, annotations, and export.
- **Seaborn:** statistical visualizations and attractive defaults, built on Matplotlib.
- **Plotly Express:** interactive charts with hover details, zooming, and dashboard-friendly output.
- **Cufflinks:** a convenience wrapper historically used for interactive Pandas plots; it is optional and not required here.

The examples below use Matplotlib and Seaborn as the main static plotting tools and Plotly when available.

In [ ]:
# If a package is missing, install it in your notebook environment:
# %pip install pandas numpy matplotlib seaborn plotly openpyxl

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly is not installed. Static examples will still work.")

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("Plotly available:", PLOTLY_AVAILABLE)

## 3. Create sample datasets

We will use three small datasets:
1. **Sales data** — product, region, channel, revenue, units, and date.
2. **Employee data** — department, salary, experience, performance, and gender.
3. **Marketing data** — spend by channel and revenue over time.

The random seed makes the examples reproducible.

In [ ]:
rng = np.random.default_rng(42)

# Sales dataset: row-level transactions
n = 500
dates = pd.date_range("2025-01-01", periods=12, freq="MS")
sales = pd.DataFrame({
    "Order_Date": rng.choice(dates, size=n),
    "Product": rng.choice(["Laptop", "Phone", "Tablet", "Monitor", "Accessory"], size=n,
                          p=[.18, .30, .15, .14, .23]),
    "Region": rng.choice(["North", "South", "East", "West"], size=n),
    "Channel": rng.choice(["Online", "Retail", "Partner"], size=n),
    "Units": rng.integers(1, 15, size=n),
    "Unit_Price": rng.choice([25, 50, 100, 250, 500, 900], size=n)
})
sales["Revenue"] = sales["Units"] * sales["Unit_Price"] * rng.uniform(.85, 1.15, size=n)
sales["Revenue"] = sales["Revenue"].round(2)
sales["Order_Date"] = pd.to_datetime(sales["Order_Date"])
sales["Month"] = sales["Order_Date"].dt.to_period("M").dt.to_timestamp()

# Employee dataset
m = 240
employee = pd.DataFrame({
    "Employee_ID": np.arange(1001, 1001 + m),
    "Department": rng.choice(["Sales", "IT", "Finance", "HR", "Operations"], size=m),
    "Gender": rng.choice(["Female", "Male"], size=m),
    "Experience_Years": np.clip(rng.normal(5.5, 3, size=m), 0, 18).round(1),
    "Monthly_Salary": np.clip(rng.normal(52000, 16000, size=m), 22000, 120000).round(0),
    "Performance_Score": np.clip(rng.normal(3.5, .7, size=m), 1, 5).round(1)
})
employee["Salary_Band"] = pd.cut(
    employee["Monthly_Salary"],
    bins=[0, 35000, 55000, 80000, np.inf],
    labels=["Entry", "Mid", "Senior", "Lead"]
)

# Marketing dataset
marketing = pd.DataFrame({
    "Month": dates,
    "TV_Spend": [18, 20, 19, 23, 24, 27, 26, 29, 31, 30, 34, 36],
    "Radio_Spend": [8, 9, 8, 10, 11, 10, 12, 12, 13, 14, 14, 15],
    "Digital_Spend": [12, 13, 15, 14, 17, 18, 20, 21, 23, 24, 26, 28],
})
marketing["Revenue"] = (
    120 + 4.2 * marketing["TV_Spend"] +
    2.5 * marketing["Radio_Spend"] +
    3.1 * marketing["Digital_Spend"] +
    rng.normal(0, 8, len(marketing))
).round(2)

display(sales.head())
display(employee.head())
display(marketing.head())

### Load the Datasets

You can use either the datasets generated in the previous cell or read the Excel file uploaded to the classroom. If you have downloaded the file locally, specify its path below.

```python
import pandas as pd

file = r"C:\Users\YourName\Downloads\generated_datasets.xlsx"

sales = pd.read_excel(file, sheet_name="Sales")
employee = pd.read_excel(file, sheet_name="Employee")
marketing = pd.read_excel(file, sheet_name="Marketing")

display(sales.head())
display(employee.head())
display(marketing.head())
```

**Note:** If you have already executed the data-generation cell above, you can directly use the existing `sales`, `employee`, and `marketing` DataFrames without loading the file again. Otherwise, use the file-reading code and update the path according to your local file location.

In [ ]:
# Write Above code and try youself.

## 4. Prepare and inspect data before plotting

A good plotting workflow starts with checking column names, types, missing values, duplicates, and summary statistics.

In [ ]:
print("Sales shape:", sales.shape)
print("\nColumn types:")
print(sales.dtypes)
print("\nMissing values:")
display(sales.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", sales.duplicated().sum())
display(sales.describe(include="all").T)

## 5. Basic plot anatomy

A plot has several parts. Students should learn these names early because these terms appear repeatedly in Matplotlib and Seaborn.

16. **Figure** — the full chart area  
17. **Axes** — the actual plotting area  
18. **X-axis** — horizontal axis  
19. **Y-axis** — vertical axis  
20. **Title** — what the chart is about  
21. **Labels** — explanation of each axis  
22. **Legend** — explanation of colors or groups  
23. **Grid** — reference lines  
24. **Ticks** — values shown on axes  
25. **Figure size** — width and height  
26. **Rotation of labels** — useful for long category names  
27. **Tight layout** — avoids cutting labels

In [ ]:
monthly_revenue = sales.groupby("Month", as_index=False)["Revenue"].sum()

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(monthly_revenue["Month"], monthly_revenue["Revenue"], marker="o", color="red", linewidth=1, markersize=6, label="Revenue", alpha=0.8, linestyle="-", markerfacecolor="blue", markeredgewidth=1, markeredgecolor="black", )
ax.set_title("Monthly Revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (sample currency units)")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Pandas plotting: fast exploratory charts

Pandas plotting is useful for quick EDA. For deeper styling, use the returned Matplotlib axes or create the chart directly with Matplotlib/Seaborn.

In [ ]:
# 6A. Histogram of revenue
sales["Revenue"].plot(kind="hist", bins=20, title="Transaction Revenue Distribution",
                      edgecolor="black", figsize=(8, 4))
plt.xlabel("Revenue")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.show()

# 6B. Bar chart: revenue by product
sales.groupby("Product")["Revenue"].sum().sort_values().plot(
    kind="barh", title="Total Revenue by Product", figsize=(8, 4)
)
plt.xlabel("Total revenue")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

# 6C. Line chart: monthly revenue
monthly_revenue.set_index("Month")["Revenue"].plot(
    kind="line", marker="o", title="Monthly Revenue Trend", figsize=(9, 4)
)
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

## 7. Comparison charts

Use bar/column charts to compare categories. Sorting bars makes rankings easier to read; horizontal bars work well when category names are long.

In [ ]:
# 7A. Sorted vertical bar: revenue by product
product_revenue = sales.groupby("Product")["Revenue"].sum().sort_values(ascending=False)
ax = product_revenue.plot(kind="bar", figsize=(9, 4), title="Revenue Ranking by Product")
ax.set_xlabel("Product")
ax.set_ylabel("Revenue")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

# 7B. Horizontal bar: top 10 employees by salary
top_salary = employee.nlargest(10, "Monthly_Salary").sort_values("Monthly_Salary")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_salary["Employee_ID"].astype(str), top_salary["Monthly_Salary"])
ax.set_title("Top 10 Employees by Monthly Salary")
ax.set_xlabel("Monthly salary")
ax.set_ylabel("Employee ID")
plt.tight_layout()
plt.show()

# 7C. Department-wise average salary
dept_salary = employee.groupby("Department")["Monthly_Salary"].mean().sort_values(ascending=False)
sns.barplot(x=dept_salary.index, y=dept_salary.values)
plt.title("Average Monthly Salary by Department")
plt.xlabel("Department")
plt.ylabel("Average monthly salary")
plt.tight_layout()
plt.show()

## 8. Count plots and grouped comparisons

A count plot shows the number of records in each category. Adding `hue` allows a second categorical variable to be compared.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.countplot(data=employee, x="Department", hue="Gender", ax=ax)
ax.set_title("Employee Count by Department and Gender")
ax.set_xlabel("Department")
ax.set_ylabel("Employee count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Grouped bar chart: revenue by region and sales channel
region_channel = sales.pivot_table(
    index="Region", columns="Channel", values="Revenue", aggfunc="sum", fill_value=0
)
region_channel.plot(kind="bar", figsize=(10, 5), title="Revenue by Region and Channel")
plt.ylabel("Revenue")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Trend charts

Line charts are appropriate when the x-axis has a meaningful order, particularly time. Aggregate to the required time grain before plotting.

In [ ]:
# 9A. Revenue trend
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(monthly_revenue["Month"], monthly_revenue["Revenue"], marker="o", linewidth=2)
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# 9B. Multiple marketing channel spend trends
fig, ax = plt.subplots(figsize=(10, 5))
for col in ["TV_Spend", "Radio_Spend", "Digital_Spend"]:
    ax.plot(marketing["Month"], marketing[col], marker="o"label=col.replace("_", " " )
ax.set_title("Marketing Spend by Channel Over Time")
ax.set_xlabel("Month")
ax.set_ylabel("Spend (sample units)")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 10. Distribution charts

- **Histogram:** frequency across numeric bins.
- **KDE:** smoothed estimate of the distribution shape.
- **Box plot:** median, quartiles, spread, and potential outliers.
- **Violin plot:** distribution shape across categories.
- **ECDF:** cumulative proportion at or below each value.

A distribution chart helps assess skew, spread, outliers, and differences between groups.

In [ ]:
# 10A. Histogram with KDE
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(data=employee, x="Monthly_Salary", bins=20, kde=True, ax=ax)
ax.set_title("Monthly Salary Distribution")
ax.set_xlabel("Monthly salary")
plt.tight_layout()
plt.show()

# 10B. Box plot by department
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=employee, x="Department", y="Monthly_Salary", ax=ax)
ax.set_title("Salary Spread by Department")
ax.set_xlabel("Department")
ax.set_ylabel("Monthly salary")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 10C. Violin plot by department
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=employee, x="Department", y="Monthly_Salary", inner="quartile", ax=ax)
ax.set_title("Salary Distribution by Department")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 10D. ECDF plot
fig, ax = plt.subplots(figsize=(8, 4))
sns.ecdfplot(data=employee, x="Experience_Years", ax=ax)
ax.set_title("Cumulative Distribution of Experience")
ax.set_xlabel("Experience (years)")
ax.set_ylabel("Proportion at or below value")
plt.tight_layout()
plt.show()

## 11. Relationship charts

Use scatter plots for two numeric variables. A regression plot adds a fitted trend line, but correlation does not prove causation.

In [ ]:
# 11A. Scatter plot: experience vs salary
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=employee, x="Experience_Years", y="Monthly_Salary",
    hue="Department", alpha=0.75, ax=ax
)
ax.set_title("Experience vs Monthly Salary")
ax.set_xlabel("Experience (years)")
ax.set_ylabel("Monthly salary")
plt.tight_layout()
plt.show()

# 11B. Regression plot: marketing spend vs revenue
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(data=marketing, x="Digital_Spend", y="Revenue", ax=ax)
ax.set_title("Digital Marketing Spend vs Revenue")
ax.set_xlabel("Digital spend")
ax.set_ylabel("Revenue")
plt.tight_layout()
plt.show()

# 11C. Bubble chart: units, unit price, and revenue
fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(sales["Units"], sales["Unit_Price"], s=sales["Revenue"] / 5,
                alpha=0.35)
ax.set_title("Sales Transactions: Units vs Unit Price")
ax.set_xlabel("Units")
ax.set_ylabel("Unit price")
plt.tight_layout()
plt.show()

## 12. Correlation heatmap and pair plot

Correlation measures the strength and direction of linear association between numeric variables. It is not a causal measure.

In [ ]:
numeric_marketing = marketing[["TV_Spend", "Radio_Spend", "Digital_Spend", "Revenue"]]
corr = numeric_marketing.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Marketing Variables: Correlation Heatmap")
plt.tight_layout()
plt.show()

# Pair plot: useful for exploring several numeric variables together
sns.pairplot(
    employee[["Experience_Years", "Monthly_Salary", "Performance_Score"]],
    corner=True, diag_kind="hist"
)
plt.show()

## 13. Composition charts

Composition charts explain how a total is split into parts. Stacked bars are often easier to compare than pie charts when there are multiple groups. Use a pie chart only when there are a few categories and the total is meaningful.

In [ ]:
# 13A. Stacked bar chart: channel revenue mix by region
region_channel.plot(
    kind="bar", stacked=True, figsize=(9, 5),
    title="Revenue Composition by Region"
)
plt.xlabel("Region")
plt.ylabel("Revenue")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 13B. 100% stacked bar chart: channel share by region
region_share = region_channel.div(region_channel.sum(axis=1), axis=0) * 100
region_share.plot(
    kind="bar", stacked=True, figsize=(9, 5),
    title="Percentage Revenue Mix by Region"
)
plt.ylabel("Share of regional revenue (%)")
plt.xlabel("Region")
plt.xticks(rotation=0)
plt.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# 13C. Pie chart: sales transaction count by channel
channel_counts = sales["Channel"].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(channel_counts, labels=channel_counts.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Transactions by Channel")
ax.axis("equal")
plt.show()

## 14. Geographical views

A true map requires geographic information such as latitude/longitude or a supported boundary dataset. The sample data only contains broad region labels, so a regional comparison chart is the honest choice here—not a fabricated map.

In [ ]:
regional_summary = sales.groupby("Region", as_index=False).agg(
    Revenue=("Revenue", "sum"),
    Orders=("Revenue", "size")
).sort_values("Revenue", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=regional_summary, x="Region", y="Revenue", ax=ax)
ax.set_title("Revenue by Region (Regional Comparison)")
ax.set_xlabel("Region")
ax.set_ylabel("Revenue")
plt.tight_layout()
plt.show()

display(regional_summary)

In [ ]:
%pip install "nbformat>=4.2.0"
%pip install "nbformat>=4.2.0"

In [ ]:
%pip install plotly

## 15. Interactive charts with Plotly

Plotly charts support hover details, zooming, and interactive legends. If Plotly is unavailable, skip this section or install it using `%pip install plotly`.

In [ ]:
if PLOTLY_AVAILABLE:
    fig = px.line(
        monthly_revenue, x="Month", y="Revenue", markers=True,
        title="Interactive Monthly Revenue Trend"
    )
else:
    print("Install Plotly to run this cell: %pip install plotly")

In [ ]:
if PLOTLY_AVAILABLE:
    fig = px.bar(
        product_revenue.rename_axis("Product").reset_index(name="Revenue"),
        x="Product", y="Revenue", color="Product",
        title="Interactive Revenue by Product"
    )
    fig.show()
else:
    print("Plotly is not installed.")

In [ ]:
if PLOTLY_AVAILABLE:
    fig = px.scatter(
        employee, x="Experience_Years", y="Monthly_Salary",
        color="Department", size="Performance_Score",
        hover_data=["Employee_ID", "Gender"],
        title="Interactive Employee Experience vs Salary"
    )
    fig.show()
else:
    print("Plotly is not installed.")

## 16. Formatting charts for reports

Use informative titles, explicit units, readable labels, restrained legends, and annotations that clarify the message. Avoid unnecessary 3D effects and too many colors.

In [ ]:
product_revenue = sales.groupby("Product")["Revenue"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(product_revenue.index, product_revenue.values)
ax.set_title("Product Revenue — Highest to Lowest")
ax.set_xlabel("Product")
ax.set_ylabel("Revenue (sample units)")
ax.tick_params(axis="x", rotation=20)

# Label the bars with rounded values
ax.bar_label(bars, labels=[f"{v:,.0f}" for v in product_revenue.values], padding=3, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(y=0.15)
fig.tight_layout()
plt.show()

## 17. Save charts and export summary tables

`savefig` saves the current Matplotlib figure. Prefer a tight bounding box so labels are not clipped. You can export summary tables to CSV or Excel.

In [ ]:
output_dir = Path("plotting_outputs")
output_dir.mkdir(exist_ok=True)

# Create and save a chart
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(monthly_revenue["Month"], monthly_revenue["Revenue"], marker="o")
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(output_dir / "monthly_revenue_trend.png", dpi=150, bbox_inches="tight")
plt.show()

# Export a summary table
product_revenue.rename("Revenue").to_csv(output_dir / "revenue_by_product.csv")
regional_summary = sales.groupby("Region", as_index=False).agg(
    Revenue=("Revenue", "sum"),
    Orders=("Revenue", "size")
).sort_values("Revenue", ascending=False)

regional_summary.to_csv(output_dir / "regional_summary.csv", index=False)

print("Saved outputs to:", output_dir.resolve())

## 18. Mini EDA workflow

Use this sequence for a new dataset:
1. Load and inspect the data.
2. Validate data types, missing values, duplicates, and ranges.
3. Define the business question.
4. Aggregate to the correct grain.
5. Select a chart based on the question and variable types.
6. Add a title, labels, units, and useful context.
7. Check whether the chart could mislead.
8. Save the chart and summarize the insight in plain language.

In [ ]:
# Example mini workflow on the sales dataset
print("Rows, columns:", sales.shape)
print("Missing cells:", int(sales.isna().sum().sum()))
print("Duplicate rows:", int(sales.duplicated().sum()))

kpis = pd.Series({
    "Total Revenue": sales["Revenue"].sum(),
    "Total Units": sales["Units"].sum(),
    "Average Transaction Revenue": sales["Revenue"].mean(),
    "Number of Transactions": len(sales),
})
display(kpis.to_frame("Value"))

# Question: Which products contribute most revenue?
display(product_revenue.to_frame("Revenue"))

## 19. Common plotting mistakes

- Using a line chart for unordered categories.
- Comparing categories without sorting when ranking matters.
- Using a pie chart with many slices or comparing multiple pies.
- Forgetting units, labels, or a descriptive title.
- Plotting raw rows when the question requires aggregation.
- Ignoring missing values, duplicates, or incorrect data types.
- Treating correlation as proof of causation.
- Using a map when no geographic data exists.
- Overloading charts with colors, labels, and legends.
- Saving a chart before calling `tight_layout()` or using `bbox_inches="tight"`.

## 20. Quick reference

| Chart | Best for | Watch out for |
|---|---|---|
| Bar / column | Category comparison | Unsorted bars can hide ranking |
| Line | Trends over ordered time | Do not connect unrelated categories |
| Histogram | Numeric distribution | Bin size affects appearance |
| KDE | Smooth distribution shape | Can hide details in small samples |
| Box plot | Spread and potential outliers | Does not show every data point |
| Violin plot | Distribution shape by group | Can be harder for beginners to read |
| Scatter | Relationship between numeric variables | Overplotting and confounding |
| Heatmap | Matrix patterns / correlations | Correlation is not causation |
| Stacked bar | Composition across groups | Small segments can be hard to compare |
| Pie | Simple part-to-whole composition | Avoid many slices |
| Count plot | Frequency of categories | Check missing category values |
| Interactive Plotly | Exploration and hover detail | Ensure the interactive output is supported by the viewer |